# Keplerv · Photos to 3D vehicle

## Run all

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all**.
3. When prompted, paste your ngrok authtoken. It stays in this Colab runtime, not in this notebook or GitHub.
4. Wait for **API ready** in the last code cell. Open **AI 3D** on keplerv.com and generate a vehicle.

**Fixed website endpoint:** `https://aloha-anytime-zodiac.ngrok-free.dev`

First setup downloads a large model and can take several minutes. The last code cell deliberately keeps running to serve the website. Run all will therefore remain busy while the API is available. To rerun everything, stop execution first.

The model uses up to four separate exterior views of the same car or pickup. The output is an orbitable, untextured GLB with inferred geometry; exact dimensions and unseen details are not guaranteed. Colab may end the runtime, so this is not permanent hosting.

---


In [ ]:
# 1. Install the official Tencent implementation and shape-generation dependencies.
# First setup may take several minutes. No CUDA texture renderer is needed.
%pip install -q "git+https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git" "rembg[cpu]" scikit-image pyngrok requests "pillow==11.3.0" "uvicorn[standard]" fastapi accelerate safetensors pyyaml
# Colab may preload Pillow before pip replaces it. Drop those cached imports
# before loading any model/image modules, so PIL uses one installed version.
import sys
import importlib
for module_name in list(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        del sys.modules[module_name]
importlib.invalidate_caches()
from PIL import Image, ImageFont
print("Pillow imports ready:", Image.__version__)


In [ ]:
# 2. Load Hunyuan3D without duplicate CPU weight copies (Colab RAM is limited).
import gc
import torch
import yaml
from accelerate import init_empty_weights
from safetensors import safe_open
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.shapegen.pipelines import instantiate_from_config
from hy3dgen.shapegen.utils import smart_load_model
from rembg import new_session, remove

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU."
for old_name in ("pipe", "components", "component", "state", "tensor"):
    globals().pop(old_name, None)
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "tencent/Hunyuan3D-2mv"
config_path, checkpoint_path = smart_load_model(
    MODEL_ID, subfolder="hunyuan3d-dit-v2-mv", use_safetensors=True, variant="fp16",
)
with open(config_path) as config_file:
    config = yaml.safe_load(config_file)

# The upstream loader builds full float32 CPU modules before copying weights.
# Build empty modules and stream each checkpoint tensor directly to the GPU.
components = {}
with safe_open(checkpoint_path, framework="pt", device="cpu") as checkpoint:
    all_keys = list(checkpoint.keys())
    for name in ("model", "vae", "conditioner"):
        with init_empty_weights():
            component = instantiate_from_config(config[name])
        state = {}
        prefix = name + "."
        for key in all_keys:
            if key.startswith(prefix):
                tensor = checkpoint.get_tensor(key)
                dtype = torch.float16 if tensor.is_floating_point() else tensor.dtype
                state[key[len(prefix):]] = tensor.to(device="cuda", dtype=dtype)
        if not state:
            raise RuntimeError(f"Checkpoint is missing the {name} weights.")
        missing, unexpected = component.load_state_dict(state, strict=False, assign=True)
        # The release omits VAE encoder/pre_kl weights; only its decoder is used.
        required_missing = [key for key in missing if name != "vae" or not key.startswith(("encoder.", "pre_kl."))]
        if required_missing:
            raise RuntimeError(f"Missing required checkpoint tensors in {name}: {required_missing[:3]}")
        if name == "vae":
            component.encoder = torch.nn.Identity()
            component.pre_kl = torch.nn.Identity()
        component.to(device="cuda", dtype=torch.float16).eval()
        components[name] = component
        del state
        gc.collect()
        print("Loaded", name)

pipe = Hunyuan3DDiTFlowMatchingPipeline(
    model=components["model"], vae=components["vae"], conditioner=components["conditioner"],
    image_processor=instantiate_from_config(config["image_processor"]),
    scheduler=instantiate_from_config(config["scheduler"]),
    device="cuda", dtype=torch.float16,
)
del components
background_session = new_session("u2net", providers=["CPUExecutionProvider"])
print("Hunyuan3D multi-view model ready on", torch.cuda.get_device_name(0))


In [ ]:
# 3. Separate vehicle views -> foreground images -> one mesh.
import io
from urllib.parse import urlparse
from pathlib import Path
import requests
from PIL import Image, ImageOps
import numpy as np
import trimesh

VIEW_KEYS = ("front", "left", "back", "right")
ALLOWED_IMAGE_HOSTS = {"keplerv.com", "www.keplerv.com"}
preview_dir = Path("/content/keplerv-inputs")
preview_dir.mkdir(exist_ok=True)

def prepare_view(url: str, view: str) -> Image.Image:
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname not in ALLOWED_IMAGE_HOSTS:
        raise ValueError("Use HTTPS image URLs hosted on keplerv.com.")
    response = requests.get(url, timeout=30, allow_redirects=False)
    response.raise_for_status()
    image = ImageOps.exif_transpose(Image.open(io.BytesIO(response.content))).convert("RGB")
    image.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    foreground = remove(image, session=background_session).convert("RGBA")
    alpha = np.asarray(foreground.getchannel("A"))
    if (alpha > 127).mean() < 0.02:
        raise ValueError(f"Could not isolate the vehicle in the {view} view. Choose a clearer exterior photo.")
    # Save cutouts so the user can inspect what the geometry model actually sees.
    foreground.save(preview_dir / f"{view}.png")
    return foreground

def reconstruct_to_glb(image_urls: list[str], steps: int = 30, views: dict[str, str] | None = None, seed: int = 12345) -> bytes:
    if views is None:
        views = dict(zip(VIEW_KEYS, image_urls))
    if not views or set(views) - set(VIEW_KEYS):
        raise ValueError("Provide exterior views using front, left, back, right keys.")
    if len(set(views.values())) != len(views):
        raise ValueError("Use a different photo for each view.")
    images = {key: prepare_view(views[key], key) for key in VIEW_KEYS if key in views}
    with torch.inference_mode():
        mesh = pipe(
            image=images, num_inference_steps=steps, guidance_scale=5.0,
            octree_resolution=256, num_chunks=8000,
            generator=torch.Generator(device="cuda").manual_seed(seed),
            output_type="trimesh",
        )[0]
    if mesh is None or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise ValueError("The model returned empty geometry. Try clearer exterior views.")
    # Normalize to a predictable scale for the website's orbit controls.
    mesh.apply_translation(-mesh.bounds.mean(axis=0))
    extent = float(np.max(mesh.extents))
    if extent > 0:
        mesh.apply_scale(2.0 / extent)
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=[190, 196, 205, 255])
    return mesh.export(file_type="glb")


In [ ]:
# 4. ngrok auth — saved only inside this Colab runtime, never in GitHub.
from getpass import getpass
from pathlib import Path
from pyngrok import ngrok, conf
import yaml

config_file = Path(conf.get_default().config_path or (Path.home() / ".config/ngrok/ngrok.yml"))
conf.get_default().config_path = str(config_file)
saved_config = yaml.safe_load(config_file.read_text()) if config_file.exists() else {}
saved_token = (saved_config or {}).get("authtoken") or (saved_config or {}).get("agent", {}).get("authtoken")
if saved_token:
    conf.get_default().auth_token = saved_token
    print("Using the ngrok token saved in this runtime.")
else:
    token = getpass("Paste your ngrok authtoken: ").strip()
    if not token:
        raise ValueError("An ngrok authtoken is required. Rerun this cell and paste your token.")
    ngrok.set_auth_token(token)
    conf.get_default().auth_token = token
    del token
    print("Token saved in this runtime; kernel restarts can reuse it.")
del saved_token, saved_config


In [ ]:
# 5. API server and public tunnel; leave this cell running.
import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel, Field, model_validator

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://www.keplerv.com", "https://keplerv.com", "http://localhost:4399", "http://localhost:3000"],
    allow_methods=["GET", "POST"], allow_headers=["Content-Type", "ngrok-skip-browser-warning"],
)
gpu_lock = threading.Lock()

class ReconstructRequest(BaseModel):
    image_urls: list[str] = Field(default_factory=list, max_length=4)
    views: dict[str, str] | None = None
    steps: int = Field(default=30, ge=20, le=60)
    seed: int = Field(default=12345, ge=0, le=2147483647)

    @model_validator(mode="after")
    def validate_views(self):
        if not self.image_urls and not self.views:
            raise ValueError("Provide at least one exterior view.")
        if self.views and (len(self.views) > 4 or set(self.views) - set(VIEW_KEYS)):
            raise ValueError("Valid view names are front, left, back, right.")
        return self

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "api_version": 2, "max_images": 4, "textured": False}

@app.post("/reconstruct")
def reconstruct(req: ReconstructRequest):
    # A T4 must not process two model generations at the same time.
    if not gpu_lock.acquire(blocking=False):
        raise HTTPException(status_code=409, detail="GPU is generating another vehicle. Please retry when it finishes.")
    try:
        glb = reconstruct_to_glb(req.image_urls, steps=req.steps, views=req.views, seed=req.seed)
        return Response(content=glb, media_type="model/gltf-binary")
    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        raise HTTPException(status_code=503, detail="GPU ran out of memory. Restart the GPU runtime and rerun the notebook.") from e
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e)) from e
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e)) from e
    finally:
        gpu_lock.release()

# Reuse an existing tunnel on port 8000 when upgrading the running notebook.
tunnel = next((t for t in ngrok.get_tunnels() if str(t.config.get("addr", "")).rstrip("/") in {"http://localhost:8000", "localhost:8000", "8000"}), None)
if tunnel is None:
    tunnel = ngrok.connect(8000, "http", domain="aloha-anytime-zodiac.ngrok-free.dev")
public_url = tunnel.public_url
print("\n✅ API ready:", public_url)
print("Health check:", public_url + "/health")
print("Leave this cell running. The website already uses this address.")
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio"))
await server.serve()


### Inspect the inputs before judging the result

- The background cutouts used by the model are saved in `/content/keplerv-inputs`. Open Colab's Files panel to inspect them; a missing wheel or pickup bed in the cutout will hurt the result.
- Use the same car in every view, with the full vehicle visible and consistent camera elevation. Avoid interior, engine, and close-up damage photos.
- The camera slots are relative to the reference image and proceed clockwise. Approximate auction angles may still distort the inferred geometry.
- A silver/gray output is expected: this configuration prioritizes geometry on a T4. It does not run the separate texture-generation model.
- Shape quality is not guaranteed. Hunyuan3D infers unseen details; it is not a measurement or damage-assessment tool.
- Cell 5 stays running while serving requests. Stop it before editing/reloading the models.
